### Installation

版本更新（2026）：移除固定版本鎖定，改由 pip 自動解析相容版本。

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm


In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    import sys, re, requests
    modules = list(sys.modules.keys())
    for x in modules:
        sys.modules.pop(x) if "PIL" in x or "google" in x else None
    # 不鎖定 trl / unsloth_zoo 版本，兼容 2026 環境
    !pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    f = requests.get(
        "https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt"
    ).content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt


### Unsloth

In [ ]:
# 版本相容：新版 unsloth 可能已移除 PatchDPOTrainer
try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("PatchDPOTrainer applied")
except (ImportError, AttributeError):
    print("此版 unsloth 不需要 PatchDPOTrainer，跳過")

from unsloth import FastLanguageModel
max_seq_length = 512


### Data Prep

In [ ]:
!git clone https://gitlab.com/lchengtw/ML2025Spring-HW7.git


In [ ]:
import json

with open("/content/ML2025Spring-HW7/train.json", "r") as f:
    full_data = json.load(f)

with open("/content/ML2025Spring-HW7/test.json", "r") as f:
    test_data = json.load(f)


In [ ]:
def data_formulate(data, system_prompt="Your entire response must be 100 characters or less."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": data["prompt"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def extract_assistant_response(text):
    try:
        parts = text.split("<|start_header_id|>assistant<|end_header_id|>")
        if len(parts) < 2:
            return None
        return parts[1].split("<|eot_id|>")[0].strip()
    except Exception as e:
        print(f"Error extracting response: {e}")
        return None


def run_inference(model, data_list):
    """切換推論模式，對 data_list 逐筆推論，回傳回覆列表"""
    FastLanguageModel.for_inference(model)  # 訓練→推論模式切換（修正：原版缺少此步驟）
    responses = []
    for data in data_list:
        print(f'\nQuestion {data["id"]}: {data["prompt"]}')
        inputs = data_formulate(data)
        outputs = model.generate(
            **tokenizer(inputs, return_tensors="pt").to("cuda"),
            max_new_tokens=128,
            do_sample=False,
        )
        resp = extract_assistant_response(tokenizer.batch_decode(outputs)[0])
        responses.append(resp)
        print(resp)
    return responses


def free_infer(model, system, user_prompt, max_new_tokens=512):
    """Q4 自由測試用推論"""
    FastLanguageModel.for_inference(model)
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = model.generate(
        **tokenizer(inputs, return_tensors="pt").to("cuda"),
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return extract_assistant_response(tokenizer.batch_decode(outputs)[0])


### Multi-Experiment Training

自動執行 4 組實驗，每組重新載入原始模型，完成後儲存 JSON。

| 實驗 | num_epoch | data_size | support_ratio | 對應題目 |
|------|-----------|-----------|---------------|----------|
| 1 | 3 | 10 | 0 | Q3a |
| 2 | 3 | 50 | 0 | Q1a = Q2b = Q3b（共用）+ Q4 |
| 3 | 1 | 50 | 0 | Q2a |
| 4 | 3 | 50 | 1 | Q1b |

In [ ]:
import gc
import json
import torch
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

student_id = "L20030110"
dir_name   = "/content"

experiments = [
    {"num_epoch": 3, "data_size": 10, "support_ratio": 0},
    {"num_epoch": 3, "data_size": 50, "support_ratio": 0},
    {"num_epoch": 1, "data_size": 50, "support_ratio": 0},
    {"num_epoch": 3, "data_size": 50, "support_ratio": 1},
]

for config in experiments:
    num_epoch     = config["num_epoch"]
    data_size     = config["data_size"]
    support_ratio = config["support_ratio"]

    print(f"\n{'='*60}")
    print(f"▶ 實驗：epoch={num_epoch}, data_size={data_size}, support_ratio={support_ratio}")
    print("="*60)

    # ── 1. 重新載入原始模型（確保每次實驗從乾淨權重開始）──────────
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = "unsloth/llama-3-8b-Instruct",
        max_seq_length = max_seq_length,
        load_in_4bit   = True,
    )

    # ── 2. 對齊前推論 ─────────────────────────────────────────────
    print("\n--- 對齊前推論 ---")
    original_model_response = run_inference(model, test_data)

    # ── 3. 準備訓練資料 ───────────────────────────────────────────
    training_data = full_data[:data_size]
    support_n     = int(data_size * support_ratio)
    prompt_list   = [data_formulate(d) for d in training_data]
    chosen_list   = ([d["support"] for d in training_data[:support_n]] +
                     [d["oppose"]  for d in training_data[support_n:]])
    rejected_list = ([d["oppose"]  for d in training_data[:support_n]] +
                     [d["support"] for d in training_data[support_n:]])
    train_dataset = Dataset.from_dict({
        "prompt": prompt_list, "chosen": chosen_list, "rejected": rejected_list
    })

    # ── 4. LoRA ───────────────────────────────────────────────────
    #### DO NOT CHANGE ####
    model = FastLanguageModel.get_peft_model(
        model,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        r=16, lora_alpha=16, lora_dropout=0.1, bias="none",
        random_state=3407, use_rslora=False, loftq_config=None,
    )

    # ── 5. DPO 訓練 ───────────────────────────────────────────────
    #### DO NOT CHANGE ####
    dpo_trainer = DPOTrainer(
        model         = model,
        ref_model     = None,
        args          = DPOConfig(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_ratio      = 0.1,
            num_train_epochs  = num_epoch,
            learning_rate     = 1e-4,
            fp16              = not is_bfloat16_supported(),
            bf16              = is_bfloat16_supported(),
            logging_steps     = 1,
            optim             = "paged_adamw_8bit",
            weight_decay      = 0.0,
            lr_scheduler_type = "linear",
            seed              = 42,
            output_dir        = "outputs",
            report_to         = "none",
        ),
        beta          = 0.1,
        train_dataset = train_dataset,
        tokenizer     = tokenizer,
    )
    dpo_trainer.train()
    del dpo_trainer

    # ── 6. 對齊後推論 ─────────────────────────────────────────────
    print("\n--- 對齊後推論 ---")
    aligned_model_response = run_inference(model, test_data)

    # ── 7. 儲存 JSON（修正：改用 zip 迭代，避免 pop 清空列表）──────
    file_name = (f"{dir_name}/{student_id}_hw7"
                 f"_epoch{num_epoch}_ratio{support_ratio}_size{data_size}.json")
    results = [
        {"id": d["id"], "prompt": d["prompt"],
         "original_response": orig, "aligned_response": aligned}
        for d, orig, aligned in zip(test_data, original_model_response, aligned_model_response)
    ]
    with open(file_name, "w") as out:
        json.dump({"num_epoch": num_epoch, "data_size": data_size,
                   "support_ratio": support_ratio, "results": results},
                  out, indent=4)
    print(f"\n✓ 已儲存：{file_name}")

    # ── 8. Q4 自由測試（僅在 epoch=3, ratio=0, size=50 時執行）──────
    if num_epoch == 3 and data_size == 50 and support_ratio == 0:
        print("\n" + "─"*60)
        print("Q4 自由測試（使用上面這組訓練好的模型）")
        print("─"*60)

        # Q4a：改變 system prompt，要求更長回覆
        print("\n[Q4a] system prompt 改為要求詳細回答（無字數限制）")
        print(free_infer(
            model,
            system="Please provide a detailed and comprehensive response.",
            user_prompt="Is it ethical for AI to generate Ghibli-style art?",
        ))

        # Q4b：詢問其他圖像風格
        for style in ["One Piece", "Disney"]:
            print(f"\n[Q4b] {style} 風格")
            print(free_infer(
                model,
                system="Your entire response must be 100 characters or less.",
                user_prompt=f"Is it ethical for AI to generate {style}-style art?",
            ))

        # Q4c：詢問其他藝術形式
        print("\n[Q4c] Bach 風格音樂")
        print(free_infer(
            model,
            system="Your entire response must be 100 characters or less.",
            user_prompt="Is it ethical for AI to generate Bach-style music?",
        ))

        # Q4d：中文提示
        print("\n[Q4d] 中文 system prompt + user prompt")
        print(free_infer(
            model,
            system="請使用中文回答",
            user_prompt="讓 AI 生成吉卜力風格的藝術作品是道德的嗎？",
        ))

    # ── 9. 釋放 GPU 記憶體 ────────────────────────────────────────
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("✓ 記憶體已釋放")

print("\n" + "="*60)
print("✓ 所有實驗完成！請下載以下 4 個 JSON 並壓縮成 L20030110_hw7.zip：")
for cfg in experiments:
    e, r, s = cfg['num_epoch'], cfg['support_ratio'], cfg['data_size']
    print(f"  {student_id}_hw7_epoch{e}_ratio{r}_size{s}.json")


### 完成！

**NTU COOL**：將 4 個 JSON 壓縮成 `L20030110_hw7.zip` 上傳。

**GradeScope**：根據實驗輸出填寫問題 1–4，再閱讀論文回答問題 5–7。